In [15]:
import json
import requests
import pandas as pd
import chromadb

from pathlib import Path
from chromadb.utils import embedding_functions

In [16]:
RESULTS_DIR = Path("../data/results")
ATTCK_DIR = Path("../data/attck")
CHROMA_DIR = Path("../data/chroma")

COMMUNITY_FILE = RESULTS_DIR / "community_assignments.csv"
TRIPLES_FILE = RESULTS_DIR / "community_triples.json"
STIX_FILE = ATTCK_DIR / "enterprise-attack.json"

REPORTS_FILE = RESULTS_DIR / "stage5_rag_reports.csv"
REPORT_METRICS_FILE = RESULTS_DIR / "stage5_rag_metrics.json"

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen2.5:3b"
TIMEOUT = 180

TOP_K = 3
MAX_COMMUNITIES = 20
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"

In [17]:
# Load data
community_df = pd.read_csv(COMMUNITY_FILE, low_memory=False)

with open(TRIPLES_FILE, "r", encoding="utf-8") as f:
    community_triples = json.load(f)

print("Community rows:", len(community_df))
print("Unique communities:", community_df["community_id"].nunique())

Community rows: 10662
Unique communities: 16


In [18]:
#load ATT&CK techniques
with open(STIX_FILE, "r", encoding="utf-8") as f:
    bundle = json.load(f)

attack_docs = []

for obj in bundle.get("objects", []):
    if obj.get("type") != "attack-pattern":
        continue
    if obj.get("revoked", False) or obj.get("deprecated", False):
        continue

    technique_id = None
    for ref in obj.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            technique_id = ref.get("external_id")
            break

    if not technique_id:
        continue

    tactics = []
    for phase in obj.get("kill_chain_phases", []):
        if phase.get("kill_chain_name") == "mitre-attack":
            tactics.append(phase["phase_name"].replace("-", " ").title())

    technique_name = obj.get("name", "")
    description = obj.get("description", "")
    tactic_text = ", ".join(tactics) if tactics else "Unknown"

    doc_text = (
        f"Technique ID: {technique_id}\n"
        f"Technique Name: {technique_name}\n"
        f"Tactic: {tactic_text}\n"
        f"Description: {description}"
    )

    attack_docs.append({
        "id": technique_id,
        "text": doc_text,
        "name": technique_name,
        "tactic": tactic_text
    })

print("ATT&CK documents:", len(attack_docs))

ATT&CK documents: 703


In [19]:
# Set up ChromaDB
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBED_MODEL_NAME
)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

collection = client.get_or_create_collection(
    name="attack_techniques",
    embedding_function=ef
)

existing_count = collection.count()
print("Existing Chroma count:", existing_count)

if existing_count == 0:
    collection.add(
        ids=[doc["id"] for doc in attack_docs],
        documents=[doc["text"] for doc in attack_docs],
        metadatas=[
            {
                "technique_id": doc["id"],
                "technique_name": doc["name"],
                "tactic": doc["tactic"]
            }
            for doc in attack_docs
        ]
    )
    print("Inserted ATT&CK documents into ChromaDB")
else:
    print("Collection already populated")

Existing Chroma count: 703
Collection already populated


In [20]:
def build_retrieval_query(comm_df, community_id):
    dominant_label = comm_df["Label"].value_counts().idxmax()

    top_ports = comm_df["Destination Port"].value_counts().head(3)
    port_text = ", ".join([str(p) for p in top_ports.index])

    triples = community_triples.get(str(community_id), [])
    triple_lines = [
        f"{t['subject']} {t['relation']} {t['object']}"
        for t in triples[:6]
    ]
    triple_text = "; ".join(triple_lines)

    return (
        f"IDS label: {dominant_label}. "
        f"Top ports: {port_text}. "
        f"Observed relations: {triple_text}."
    )

def build_report_summary(comm_df, community_id):
    """
    Build report context summary from observed evidence only.
    Do NOT include ground-truth-like label fields.
    """
    top_ports = comm_df["Destination Port"].value_counts().head(3)
    port_text = ", ".join([str(p) for p in top_ports.index])

    triples = community_triples.get(str(community_id), [])
    triple_lines = [
        f"{t['subject']} {t['relation']} {t['object']}"
        for t in triples[:6]
    ]
    triple_text = "; ".join(triple_lines) if triple_lines else "none"

    return (
        f"Top destination ports: {port_text}. "
        f"Extracted relations: {triple_text}."
    )

In [21]:
def retrieve_attack_context(query_text, top_k=TOP_K):
    result = collection.query(
        query_texts=[query_text],
        n_results=top_k
    )

    docs = result["documents"][0]
    metas = result["metadatas"][0]
    return docs, metas

In [22]:
def call_ollama(prompt, model=MODEL_NAME):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0,
            "num_predict": 300
        }
    }

    response = requests.post(OLLAMA_URL, json=payload, timeout=TIMEOUT)
    response.raise_for_status()
    return response.json().get("response", "").strip()

In [23]:
def build_baseline_prompt(summary_text):
    return f"""
You are a cybersecurity analyst writing a short incident report.

Use only the community summary below.
Do not use external threat knowledge.
Do not invent unsupported evidence.
Use plain text only.
Do not use markdown or bullet points.

COMMUNITY SUMMARY:
{summary_text}

Write a short report with exactly these labels (one per line).
Each section must be 1–2 sentences.

Technique:
Summary:
Evidence:
Recommended next step:
"""

In [24]:
def build_rag_prompt(summary_text, retrieved_docs):
    context = "\n\n".join(retrieved_docs)

    return f"""
You are a cybersecurity analyst writing a short incident report.

Use only the retrieved ATT&CK context below and the community summary.
Select the single most relevant ATT&CK technique from the retrieved context.
Do not mention techniques that are not supported by the summary.
Do not speculate beyond the retrieved context.
Use plain text only.
Do not use markdown or bullet points.

COMMUNITY SUMMARY:
{summary_text}

RETRIEVED ATT&CK CONTEXT:
{context}

Write a short report with exactly these labels (one per line).

Technique:
Summary:
Evidence:
Recommended next step:
"""

In [25]:
def extract_mentioned_technique_ids(text):
    """
    simple pattern-based extraction of ATT&CK IDs from generated text.
    """
    import re
    return re.findall(r"T\d{4}(?:\.\d{3})?", text or "")

In [26]:
test_cid = community_df["community_id"].unique()[0]
test_group = community_df[community_df["community_id"] == test_cid]

test_query = build_retrieval_query(test_group, test_cid)
test_summary = build_report_summary(test_group, test_cid)
test_docs, test_meta = retrieve_attack_context(test_query, top_k=TOP_K)

print("RETRIEVAL QUERY:\n")
print(test_query)

print("\nSUMMARY:\n")
print(test_summary)

print("\nRETRIEVED TECHNIQUES:\n")
for m in test_meta:
    print(m)

test_baseline_report = call_ollama(build_baseline_prompt(test_summary))
test_rag_report = call_ollama(build_rag_prompt(test_summary, test_docs))

print("\nBASELINE REPORT:\n")
print(test_baseline_report)

print("\nRAG REPORT:\n")
print(test_rag_report)


RETRIEVAL QUERY:

IDS label: PortScan. Top ports: 444, 123, 389. Observed relations: network flow targets_service unknown service; network flow shows_behavior very short connection; network flow has_duration short duration; network flow indicates_activity port scanning activity.

SUMMARY:

Top destination ports: 444, 123, 389. Extracted relations: network flow targets_service unknown service; network flow shows_behavior very short connection; network flow has_duration short duration; network flow indicates_activity port scanning activity.

RETRIEVED TECHNIQUES:

{'technique_id': 'T1043', 'tactic': 'Command And Control', 'technique_name': 'Commonly Used Port'}
{'technique_name': 'Non-Standard Port', 'technique_id': 'T1571', 'tactic': 'Command And Control'}
{'technique_name': 'DNS Calculation', 'tactic': 'Command And Control', 'technique_id': 'T1568.003'}

BASELINE REPORT:

Technique: Port Scanning
Summary: The incident involves port scanning activities targeting ports 444, 123, and 389.

In [27]:
results = []
community_ids = (
    community_df.groupby("community_id")["attck_technique_id"]
    .agg(lambda x: x.mode().iloc[0])
)

community_ids = community_ids[community_ids != "BENIGN"].index.tolist()[:MAX_COMMUNITIES]

for cid in community_ids:
    group = community_df[community_df["community_id"] == cid]
    summary_text = build_report_summary(group, cid)
    query_text = build_retrieval_query(group, cid)

    dominant_gt = group["attck_technique_id"].mode().iloc[0]

    try:
        docs, metas = retrieve_attack_context(query_text, top_k=TOP_K)
        baseline_report = call_ollama(build_baseline_prompt(summary_text))
        rag_report = call_ollama(build_rag_prompt(summary_text, docs))
    except Exception as e:
        docs, metas = [], []
        baseline_report = f"ERROR: {e}"
        rag_report = f"ERROR: {e}"

    retrieved_ids = [m["technique_id"] for m in metas] if metas else []
    retrieved_names = [m["technique_name"] for m in metas] if metas else []

    baseline_ids = extract_mentioned_technique_ids(baseline_report)
    rag_ids = extract_mentioned_technique_ids(rag_report)

    results.append({
        "community_id": cid,
        "ground_truth": dominant_gt,
        "retrieval_query": query_text,
        "summary": summary_text,
        "retrieved_ids": ", ".join(retrieved_ids),
        "retrieved_names": ", ".join(retrieved_names),
        "ground_truth_in_retrieval": dominant_gt in retrieved_ids,
        "baseline_report": baseline_report,
        "rag_report": rag_report,
        "baseline_length": len(baseline_report),
        "rag_length": len(rag_report),
        "baseline_mentions_gt": dominant_gt in baseline_ids,
        "rag_mentions_gt": dominant_gt in rag_ids
    })

reports_df = pd.DataFrame(results)
reports_df.to_csv(REPORTS_FILE, index=False)
print(f"Saved to {REPORTS_FILE}")

Saved to ../data/results/stage5_rag_reports.csv


In [28]:
rag_metrics = {
    "n_communities_evaluated": int(len(reports_df)),
    "retrieval_hit_rate": float(reports_df["ground_truth_in_retrieval"].mean()),
    "baseline_gt_mention_rate": float(reports_df["baseline_mentions_gt"].mean()),
    "rag_gt_mention_rate": float(reports_df["rag_mentions_gt"].mean()),
    "mean_baseline_length": float(reports_df["baseline_length"].mean()),
    "mean_rag_length": float(reports_df["rag_length"].mean())
}

with open(REPORT_METRICS_FILE, "w", encoding="utf-8") as f:
    json.dump(rag_metrics, f, indent=2)

print(f"Saved to {REPORT_METRICS_FILE}")
print(rag_metrics)

reports_df.head()

Saved to ../data/results/stage5_rag_metrics.json
{'n_communities_evaluated': 13, 'retrieval_hit_rate': 0.07692307692307693, 'baseline_gt_mention_rate': 0.0, 'rag_gt_mention_rate': 0.07692307692307693, 'mean_baseline_length': 592.3076923076923, 'mean_rag_length': 601.7692307692307}


,community_id,ground_truth,retrieval_query,summary,retrieved_ids,retrieved_names,ground_truth_in_retrieval,baseline_report,rag_report,baseline_length,rag_length,baseline_mentions_gt,rag_mentions_gt
0,0,T1046,"IDS label: PortScan. Top ports: 444, 123, 389....","Top destination ports: 444, 123, 389. Extracte...","T1043, T1571, T1568.003","Commonly Used Port, Non-Standard Port, DNS Cal...",False,Technique: Port Scanning\nSummary: The inciden...,Technique: T1571\nSummary: The incident involv...,375,698,False,False
1,1,T1110.001,IDS label: FTP Patator. Top ports: 21. Observe...,Top destination ports: 21. Extracted relations...,"T1571, T1043, T1205","Non-Standard Port, Commonly Used Port, Traffic...",False,Technique: Port Scanning and FTP Service Activ...,Technique: Non-Standard Port\nSummary: The inc...,813,640,False,False
2,3,T1046,IDS label: PortScan. Top ports: 3389. Observed...,Top destination ports: 3389. Extracted relatio...,"T1070.007, T1571, T1562.004",Clear Network Connection History and Configura...,False,Technique: Port 3389 is commonly associated wi...,Technique: T1571\nSummary: Adversaries may com...,586,700,False,False
3,4,T1046,IDS label: PortScan. Top ports: 7402. Observed...,Top destination ports: 7402. Extracted relatio...,"T1043, T1571, T1205","Commonly Used Port, Non-Standard Port, Traffic...",False,Technique: Port Scanning and Brute Force Attac...,Technique: Non-Standard Port\nSummary: The inc...,740,567,False,False
4,5,T1498.001,IDS label: DDoS. Top ports: 80. Observed relat...,Top destination ports: 80. Extracted relations...,"T1498, T1499, T1016.001","Network Denial of Service, Endpoint Denial of ...",False,Technique: Port Scanning and Exploitation\n\nS...,Technique: T1498\nSummary: Network Denial of S...,598,770,False,False
